# 3D Gaussian Splatting on Tanks & Temples (“train” scene)

Training a **3D Gaussian Splatting (3DGS)** model with [Nerfstudio](https://docs.nerf.studio/)'s `splatfacto` method on a free cloud GPU (Kaggle, NVIDIA T4), then evaluating it, rendering a fly-through video and exporting the scene as a `.ply` file.

This is a first step toward **structural change detection**: once a structure is reconstructed in 3D at two dates, the two models can be rendered from identical viewpoints and compared.

**Author:** Yahya Houimdi · November 2026

## Results

| Metric (38 held-out views) | Value |
|---|---|
| PSNR | **20.78 dB** (± 3.28) |
| SSIM | **0.790** (± 0.084) |
| LPIPS | **0.203** (± 0.067) |
| Rendering speed | **~29 FPS** |
| Gaussians in the final scene | **499,057** |
| Training | 15,000 steps, ~84 ms/step → **~21 min** on one T4 |

Outputs: `eval.json` (metrics), `renders/train_flythrough.mp4` (video), `exports/splat.ply` (3D scene, ~119 MB; open it in the free [SuperSplat editor](https://superspl.at/editor)).

> These numbers are a solid first result, not state of the art: the original 3DGS paper trains for 30,000 steps on a stronger GPU.

## Pipeline

1. **Setup**: install Nerfstudio and download the official 3DGS evaluation pack (Tanks & Temples + Deep Blending, already processed with COLMAP).
2. **Fix the data format**: the images are stored at half resolution, but the COLMAP camera file describes full-size images.
3. **Train** `splatfacto` for 15,000 steps.
4. **Evaluate** on held-out views (PSNR, SSIM, LPIPS).
5. **Render** a fly-through video.
6. **Export** the Gaussians as a `.ply` file.

**Requirements:** a GPU runtime (Kaggle: *Settings → Accelerator → GPU T4*; Colab: *Runtime → Change runtime type → T4 GPU*).

## 1. Setup

In [ ]:
# Check the GPU, install Nerfstudio (~5-10 min) and download the dataset (~650 MB).
# The pip "dependency conflict" warnings about protobuf are expected and harmless here.
!nvidia-smi -L
!pip install -q nerfstudio
!test -d tandt || (wget -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip && unzip -q -o tandt_db.zip && rm -f tandt_db.zip)
!ls tandt/train

## 2. Fix the image / camera mismatch

The dataset's images are **980 × 545 px**, but `sparse/0/cameras.bin` describes the original **1959 × 1090 px** photos.
Nerfstudio checks that they match, so we tell it the images are already downscaled by 2:

- `--downscale-factor 2` halves the camera intrinsics; Nerfstudio then looks for the images in `images_2/`, so we provide that folder;
- `--downscale-rounding-mode ceil` rounds 1959 / 2 = 979.5 **up** to 980, matching the real width (the default rounds down to 979 and fails).

In [ ]:
!rm -rf tandt/train/images_2 && cp -r tandt/train/images tandt/train/images_2
!ls tandt/train

## 3. Train

- The first run compiles gsplat's CUDA kernels. On Kaggle's 4 CPUs the default parallel build filled the RAM and stalled, so `MAX_JOBS=2` limits it to two jobs. **The compile alone takes ~30 min the first time**; the GPU stays idle until it finishes.
- `--vis tensorboard` prints progress lines (the default live viewer table doesn't render in notebooks).

In [ ]:
!MAX_JOBS=2 ns-train splatfacto \
    --data tandt/train \
    --max-num-iterations 15000 \
    --viewer.quit-on-train-completion True \
    --vis tensorboard \
    colmap --colmap-path sparse/0 --downscale-factor 2 --downscale-rounding-mode ceil

## 4. Locate the trained model

Every run is saved under `outputs/train/splatfacto/<timestamp>/`. This picks the most recent one automatically.

In [ ]:
import glob, os
CONFIG = sorted(glob.glob("outputs/train/splatfacto/*/config.yml"))[-1]
os.environ["CONFIG"] = CONFIG
# PyTorch >= 2.6 loads checkpoints with weights_only=True by default, which rejects Nerfstudio's
# checkpoints (they contain NumPy scalars). This is safe ONLY for checkpoints you trained yourself.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
print("Using", CONFIG)

## 5. Evaluate on held-out views

In [ ]:
!ns-eval --load-config $CONFIG --output-path eval.json
import json
r = json.load(open("eval.json"))["results"]
print(f"PSNR  {r['psnr']:.2f} dB (± {r['psnr_std']:.2f})")
print(f"SSIM  {r['ssim']:.3f} (± {r['ssim_std']:.3f})")
print(f"LPIPS {r['lpips']:.3f} (± {r['lpips_std']:.3f})")
print(f"FPS   {r['fps']:.1f}")

## 6. Render a fly-through video

In [ ]:
!ns-render interpolate --load-config $CONFIG --output-path renders/train_flythrough.mp4 --interpolation-steps 5 --frame-rate 24
from IPython.display import Video
Video("renders/train_flythrough.mp4", embed=True, width=720)

## 7. Export the 3D scene

Writes all Gaussians to `exports/splat.ply`. **This takes several minutes and prints nothing while it runs: don't interrupt it.** The finished file is about 119 MB.

In [ ]:
!ns-export gaussian-splat --load-config $CONFIG --output-dir exports/
!ls -lh exports/

## 8. Download the results

Kaggle deletes `/kaggle/working` when the session ends, so download the results right away. Click the link printed below.

In [ ]:
!mkdir -p results && cp eval.json renders/train_flythrough.mp4 exports/splat.ply results/
!cd results && zip -q -r ../3dgs_results.zip . && cd .. && ls -lh 3dgs_results.zip
from IPython.display import FileLink
FileLink("3dgs_results.zip")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `SyntaxError` on `wget` / `pip` lines | Shell commands run as Python | Prefix them with `!` |
| `size of image (980, 545) does not match camera (1959, 1090)` | Images are half-resolution | `--downscale-factor 2` + an `images_2/` folder |
| `... does not match ((979, 545))` | 1959 / 2 rounded down | `--downscale-rounding-mode ceil` |
| `Would you like to downscale the images...? [y/n]` | `images_2/` not found | Answer **n**, then recreate `images_2/` (section 2) |
| `gsplat: Setting up CUDA...` for a very long time, RAM in the red | Parallel CUDA build fills memory | `MAX_JOBS=2` (or 1) and `rm -rf ~/.cache/torch_extensions` |
| `UnpicklingError: Weights only load failed` | PyTorch 2.6 default changed | `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` (own checkpoints only) |
| `splat.ply` much smaller than ~119 MB | Export was interrupted | Re-run section 7 and wait |

## Next steps toward change detection

- Capture the same structure at two dates (T1, T2), train one model per date, and align them (RANSAC + ICP on stable regions).
- Render both models from identical virtual cameras and compare deep features to find changed regions.
- Turn each change into a structured record and generate a grounded inspection report.

## References

- B. Kerbl, G. Kopanas, T. Leimkühler, G. Drettakis, *3D Gaussian Splatting for Real-Time Radiance Field Rendering*, ACM TOG (SIGGRAPH) 2023.
- Nerfstudio and gsplat: https://docs.nerf.studio/ · https://github.com/nerfstudio-project/gsplat
- Dataset: Tanks and Temples, as packaged by the 3DGS authors (https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/).